# Is your backtest real? Check it in 30 seconds

[TokIO](https://github.com/jordanahern2009-svg/Tokio-ai) asks whether a backtest's profit is distinguishable from luck, once two things are allowed for:

1. **Serial dependence.** Positions held for weeks make daily P&L far from independent; a plain t-test counts every bar as fresh evidence.
2. **Your search.** The Sharpe on your screen is the best of every variant you tried. The best of twenty zero-edge strategies looks good by construction.

Run the cells top to bottom. No API key, no account. The outputs saved here are from a run on 2026-10-05; yours will move a little as new days come in.

In [1]:
%pip install -q "tokio-ai>=0.5" yfinance

## 1. A grid everyone has run: SMA crossovers on SPY

Nineteen fast/slow pairs, long when fast > slow and short otherwise, on 20 years of daily SPY. The position is decided at the close and held over the next bar, so there is no lookahead.

In [2]:
import pandas as pd
import yfinance as yf
import tokio_ai

close = yf.download("SPY", period="20y", auto_adjust=True, progress=False)["Close"].squeeze()
r = close.pct_change()

pnl, positions = {}, {}
for fast in (5, 10, 20, 50):
    for slow in (50, 100, 150, 200, 250):
        if fast >= slow:
            continue
        f, s = close.rolling(fast).mean(), close.rolling(slow).mean()
        pos = (f > s).astype(float) * 2 - 1          # +1 long, -1 short
        pos = pos.where(s.notna()).shift(1)           # known at the close, held over the next bar
        name = f"sma{fast}/{slow}"
        pnl[name], positions[name] = pos * r, pos
print(len(pnl), "variants")

19 variants


## 2. The best one, the way it usually gets reported

In [3]:
best = max(pnl, key=lambda k: pnl[k].mean() / pnl[k].std())
print(best)
print(tokio_ai.check_backtest(pnl[best], positions=positions[best]))

sma50/250


SIGNIFICANT (p=0.0378, alpha=0.05).
Sharpe 0.36 annualized over 4779 bars (~19.0 years at 252/year). A plain t-test would say p=0.0571; allowing for serial dependence in the P&L, p=0.0378 before any correction for trials.
It would pass as the best of at most 1 independent trial. If you tried more than that, it's not evidence.

No costs charged; the strategy turns over 2.0x a year.
Breakeven: it stays significant up to 20.9 bps per unit traded. If your real costs, slippage included, are higher, it's not evidence.
250 bars with a missing value in some series were dropped from all of them.
The P&L's lag-1 autocorrelation is -0.10: a test that treats bars as independent overstates the uncertainty. This one doesn't.
strategy was net long on average (mean position +0.56), and the null here is zero profit, so part of this may be the market's own drift rather than timing. To test the timing, pass benchmark= the asset's per-bar returns times 0.56.
[HAC Bartlett fixed-b L=252, one-sided | tokio-

## 3. The same backtest, honestly: every variant you tried

Pass the whole grid. The verdict is about the best variant, corrected for all of them using how correlated they are, plus the probability of backtest overfitting (does the in-sample winner keep winning out of sample?).

In [4]:
print(tokio_ai.check_backtest(pnl, positions=positions))

NOT SIGNIFICANT after correcting for 19 trials (Romano-Wolf, using their correlation) (p=0.1176, alpha=0.05). Strongest variant: sma10/100.
Sharpe 0.36 annualized over 4779 bars (~19.0 years at 252/year). A plain t-test would say p=0.0602; allowing for serial dependence in the P&L, p=0.0204 before any correction for trials.
Haircut Sharpe after 19 trials: 0.21.
It would pass as the best of at most 2 independent trials. If you tried more than that, it's not evidence.

variant     Sharpe   p naive   p alone     p adj  haircut  verdict
sma10/100     0.36    0.0602    0.0204    0.1176     0.21  not significant
sma5/100      0.32    0.0812    0.0328    0.1684     0.17  not significant
sma10/200     0.35    0.0619    0.0367    0.1825     0.18  not significant
sma50/250     0.36    0.0571    0.0378    0.1851     0.18  not significant
sma5/150      0.33    0.0774    0.0377    0.1851     0.16  not significant
sma50/150     0.35    0.0645    0.0457    0.2094     0.17  not significant
sma50/200  

## 4. With trading costs

`costs=0.0005` charges 5 bps per unit of position traded. The output also gives the **breakeven cost**: the most you could pay and still have a significant result.

In [5]:
print(tokio_ai.check_backtest(pnl, positions=positions, costs=0.0005))

NOT SIGNIFICANT after correcting for 19 trials (Romano-Wolf, using their correlation) (p=0.1446, alpha=0.05). Strongest variant: sma10/100.
Sharpe 0.34 annualized over 4779 bars (~19.0 years at 252/year). A plain t-test would say p=0.0687; allowing for serial dependence in the P&L, p=0.0262 before any correction for trials.
Haircut Sharpe after 19 trials: 0.19.
It would pass as the best of at most 1 independent trial. If you tried more than that, it's not evidence.

variant     Sharpe   p naive   p alone     p adj  haircut  verdict
sma10/100     0.34    0.0687    0.0262    0.1446     0.19  not significant
sma50/250     0.36    0.0596    0.0405    0.2042     0.17  not significant
sma10/200     0.34    0.0665    0.0415    0.2052     0.16  not significant
sma5/100      0.30    0.0965    0.0450    0.2174     0.14  not significant
sma50/200     0.34    0.0677    0.0515    0.2289     0.16  not significant
sma50/150     0.34    0.0687    0.0498    0.2289     0.15  not significant
sma5/150    

## 5. Your own backtest

**From vectorbt or backtesting.py**, pass the object itself; returns and positions are read from it:

```python
pf = vbt.Portfolio.from_signals(close, entries, exits, fees=0.001)   # a whole grid works too
print(tokio_ai.check_backtest(pf))

stats = Backtest(data, MyStrategy, commission=0.001).run()
print(tokio_ai.check_backtest(stats))
```

**From anything else**, upload a CSV with one column of per-bar returns per variant (a date column is skipped), then run the cell below. If you tried more variants than you're uploading, set `TRIALS` to the real number.

In [6]:
TRIALS = None   # e.g. 40 if you tried 40 things and kept these

try:
    from google.colab import files
    name = next(iter(files.upload()))
except ImportError:
    name = input("path to your CSV: ")
df = pd.read_csv(name)
df = df.select_dtypes("number")
print(tokio_ai.check_backtest(df if df.shape[1] > 1 else df.iloc[:, 0], trials=TRIALS))

Found something odd? [Open an issue](https://github.com/jordanahern2009-svg/Tokio-ai/issues/new/choose); there's a template for statistical concerns. The false-positive rates behind every number here are measured and reproducible: [docs/calibration.md](https://github.com/jordanahern2009-svg/Tokio-ai/blob/main/docs/calibration.md).